# Machine-Translation Agents with `smolagents`

*DGT Summer School 2026 — building a translator's assistant that orchestrates CAT (Computer-Assisted Translation) tools.*

A **machine-translation (MT) agent** is a language model that can *decide when to
reach for a tool* — a term base, a translation memory (TM), an MT engine, a
quality-assurance (QA) check — the same way a human translator works inside a CAT
environment. Instead of translating in one shot, the agent reasons step by step,
calls the right aid at the right moment, and checks its own work.

This notebook builds that idea up from a single tool to a full multi-agent
translation pipeline, using `smolagents` and the OpenAI API. It is a translation
counterpart to `Introduction_to_agents.ipynb` — the agent *concepts* are the same;
the examples are all about translation.

## 1  What is a translation agent?

Three building blocks, from smallest to largest:

| Concept | In translation terms |
|---|---|
| **Tool** | One concrete action: look up a term, query the TM, translate a segment, run a QA check. |
| **Agent** | A model that plans, calls tools in sequence, reads their output, and produces a final translation. |
| **Multi-agent system** | Several specialised agents (terminology, translation, review) coordinated by an orchestrator — a small translation workflow. |

We will meet each in turn, ending with a pipeline that routes a source segment
through terminology preparation, translation, and review.

In [1]:
# !pip install -q -U smolagents openai ddgs python-dotenv
# The uv environment already provides these — this cell is here only for reference.

## 2  Setup

In [2]:
import os
import warnings
warnings.filterwarnings("ignore")

from dotenv import load_dotenv

# Load OPENAI_API_KEY from a local .env file (which must NOT be committed).
load_dotenv()
assert os.getenv("OPENAI_API_KEY"), "OPENAI_API_KEY not found - add it to your .env file."

from openai import OpenAI
from smolagents import (
    CodeAgent,
    ToolCallingAgent,
    OpenAIServerModel,
    PythonInterpreterTool,
    FinalAnswerTool,
    DuckDuckGoSearchTool,
    tool,
)
from smolagents.memory import ActionStep, FinalAnswerStep, SystemPromptStep, TaskStep

## 3  Connecting the model

`OpenAIServerModel` wraps the OpenAI Chat Completions API in the `smolagents`
`Model` interface, so the agents can talk to the model named in `.env`. We *also*
create a raw `OpenAI` client — some of our tools call the translation service
directly, which is a realistic "tool wraps an external API" pattern.

One setting below is not cosmetic. A `CodeAgent` does not use the provider's
tool-calling API: it asks the model for text shaped as `Thought: ...` followed by a
fenced Python block, and parses that text with a regex. Reasoning models
(`gpt-5.x`, `o3`, `o4`) default to `reasoning_effort="none"` here, and with
reasoning switched off they answer the task in one pass and skip the wrapper — so
the parse fails with *"Make sure to provide correct code blobs."* Asking for
`"medium"` restores it. See section 8 of `smolagents.md` for the measurements.

In [ ]:
# Model id and endpoint are read from .env (OPENAI_MODEL, OPENAI_BASE_URL),
# so the same code works against OpenAI or the course's custom endpoint.
MODEL_ID = os.getenv("OPENAI_MODEL", "gpt-5.1")
BASE_URL = os.getenv("OPENAI_BASE_URL", "https://api.openai.com/v1")
REASONING_EFFORT = os.getenv("OPENAI_REASONING_EFFORT", "medium")


model = OpenAIServerModel(
    model_id=MODEL_ID,
    api_base=BASE_URL,
    api_key=os.environ["OPENAI_API_KEY"],
    reasoning_effort=REASONING_EFFORT,  # if you use a reasoning model
)

# A CodeAgent runs the model's code in a sandboxed interpreter that refuses any
# import not on this list. Models habitually reach for `typing`, so allow the few
# harmless standard-library modules rather than spending agent steps on the error.
SAFE_IMPORTS = ["typing", "json", "textwrap"]

# A raw OpenAI client for the tools that call the MT / LLM service directly.
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=BASE_URL)

print(f"Model ready: {MODEL_ID}  (endpoint: {BASE_URL})")
print(f"Reasoning effort: {REASONING_EFFORT}")

Model ready: gpt-5.1  (endpoint: https://api.openai.com/v1)
Reasoning effort: medium


## 4  A first agent, with no tools

Given nothing but the model, an agent can already translate from its own knowledge.
We start with the simplest possible job: **one sentence, one deliverable**. Note that
even though we pass `tools=[]`, `smolagents` always adds a `final_answer` tool — so
returning the answer is itself a tool call, and the trace below shows the agent
taking exactly one step: think, act, done.

In [22]:
agent = ToolCallingAgent(
    tools=[],
    model=model,
    max_steps=4,
)

result = agent.run(
    "Translate the sentence below into English. Answer with the translation only.\n\n"
    "« La souris est sur le bureau. »"
)
print("\nFinal answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Translate the sentence below into English. Answer with the translation only.                                    │
│                                                                                                                 │
│ « La souris est sur le bureau. »                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.1 ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'The mouse is on the desk.'}                            │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: The mouse is on the desk.

Final answer: The mouse is on the desk.

[Step 1: Duration 1.27 seconds| Input tokens: 944 | Output tokens: 54]


Final answer: The mouse is on the desk.


One step, one answer — but look closely at what the agent had to guess. *Souris* is
either an animal or a computer mouse; *bureau* is either a desk or an office. The
agent picked one reading silently, and nothing in the trace tells you it made a
choice at all.

That is the whole problem: the bare agent has **no term base, no translation memory,
and no QA** — so it cannot guarantee approved terminology, reuse previous work, or
check numbers and length. In a professional workflow those guarantees are the point.
That is what tools give us.

## 5  Building the translation toolbox

We now write five custom tools with the `@tool` decorator. The decorator turns an
annotated Python function into something the agent can discover and call — the
function name, docstring, and type hints become the tool's interface.

- `machine_translate` — calls the MT engine (a live LLM call).
- `detect_language` — identifies the source language.
- `glossary_lookup` — approved translations from an IATE-style term base.
- `translation_memory_lookup` — fuzzy match against previously translated segments.
- `qa_check` — automatic quality checks on a translation.

In [41]:
@tool
def machine_translate(text: str, source_lang: str, target_lang: str) -> str:
    """Translate text from one language to another using an MT engine.

    Use this tool when an exact translation is requested rather than an explanation, summary, or language identification.

    Args:
        text: The source text to translate.
        source_lang: ISO 639-1 code of the source language (e.g. "en", "fr", "de").
        target_lang: ISO 639-1 code of the target language (e.g. "en", "fr", "de").
    """
    response = client.chat.completions.create(
        model=MODEL_ID,
        reasoning_effort="low",  # if you use a reasoning model
        messages=[
            {
                "role": "system",
                "content": (
                    "You are a professional translator. Translate the user's text "
                    f"from {source_lang} to {target_lang}. Return only the translation, "
                    "with no explanations and no surrounding quotation marks."
                ),
            },
            {"role": "user", "content": text},
        ],
    )
    return response.choices[0].message.content.strip()


# Inspect what the model will see when deciding whether to call this tool
print("Tool name  :", machine_translate.name)
print("Description:", machine_translate.description)
print("Inputs     :", machine_translate.inputs)
print("Output type:", machine_translate.output_type)

# Call it directly to confirm it works
print("\nDirect call:", machine_translate("Good morning, everyone.", "en", "de"))

Tool name  : machine_translate
Description: Translate text from one language to another using an MT engine.

Use this tool when an exact translation is requested rather than an explanation, summary, or language identification.
Inputs     : {'text': {'type': 'string', 'description': 'The source text to translate.'}, 'source_lang': {'type': 'string', 'description': 'ISO 639-1 code of the source language (e.g. "en", "fr", "de").'}, 'target_lang': {'type': 'string', 'description': 'ISO 639-1 code of the target language (e.g. "en", "fr", "de").'}}
Output type: string

Direct call: Guten Morgen, zusammen.


In [40]:
@tool
def detect_language(text: str) -> str:
    """Detect the language of a piece of text.

    Returns a two-letter ISO 639-1 code such as "en", "fr", "de" or "es".

    Args:
        text: The text whose language should be identified.
    """
    response = client.chat.completions.create(
        model=MODEL_ID,
        reasoning_effort="low",
        messages=[
            {
                "role": "system",
                "content": (
                    "Identify the language of the user's text. Answer with a single "
                    "ISO 639-1 code (e.g. en, fr, de, es) and nothing else."
                ),
            },
            {"role": "user", "content": text},
        ],
        
    )
    return response.choices[0].message.content.strip().lower()[:2]


for sample in ["Buenos días a todos.", "Guten Morgen zusammen.", "Bonjour à tous."]:
    print(f"{detect_language(sample)}  <-  {sample}")

es  <-  Buenos días a todos.
de  <-  Guten Morgen zusammen.
fr  <-  Bonjour à tous.


### 5.1  Deterministic aids: term base and translation memory

Not every tool needs the model. A **term base** (glossary) and a **translation
memory** are just data the tool looks things up in — here backed by small
in-notebook dictionaries. In production these would be IATE and Euramis; the
interface the agent sees is identical.

In [43]:
# A tiny IATE-style multilingual term base: approved translations for domain terms.
GLOSSARY = {
    "machine translation": {"fr": "traduction automatique", "de": "maschinelle Übersetzung", "es": "traducción automática"},
    "translation memory":  {"fr": "mémoire de traduction", "de": "Translation Memory", "es": "memoria de traducción"},
    "member state":        {"fr": "État membre", "de": "Mitgliedstaat", "es": "Estado miembro"},
    "regulation":          {"fr": "règlement", "de": "Verordnung", "es": "reglamento"},
    "directive":           {"fr": "directive", "de": "Richtlinie", "es": "directiva"},
}


@tool
def glossary_lookup(term: str, target_lang: str) -> dict:
    """Look up the approved translation of a term in the term base (glossary).

    Args:
        term: The source term to look up (English, case-insensitive).
        target_lang: ISO 639-1 code of the target language (e.g. "fr", "de", "es").
    """
    entry = GLOSSARY.get(term.strip().lower())
    if not entry or target_lang not in entry:
        return {"term": term, "found": False, "translation": None}
    return {"term": term, "found": True, "translation": entry[target_lang]}


print(glossary_lookup("Member State", "fr"))
print(glossary_lookup("blockchain", "fr"))   # not in the term base

{'term': 'Member State', 'found': True, 'translation': 'État membre'}
{'term': 'blockchain', 'found': False, 'translation': None}


In [44]:
from difflib import SequenceMatcher

# A tiny translation memory: previously translated segments (source -> per-language target).
TRANSLATION_MEMORY = {
    "The Commission shall adopt implementing acts.": {
        "fr": "La Commission adopte des actes d'exécution.",
        "de": "Die Kommission erlässt Durchführungsrechtsakte.",
    },
    "This Regulation shall be binding in its entirety.": {
        "fr": "Le présent règlement est obligatoire dans tous ses éléments.",
        "de": "Diese Verordnung ist in allen ihren Teilen verbindlich.",
    },
    "Member States shall bring into force the necessary provisions.": {
        "fr": "Les États membres mettent en vigueur les dispositions nécessaires.",
        "de": "Die Mitgliedstaaten setzen die erforderlichen Vorschriften in Kraft.",
    },
}


@tool
def translation_memory_lookup(segment: str, target_lang: str) -> dict:
    """Find the closest previously translated segment in the translation memory (fuzzy match).

    Returns the best matching source segment, a similarity score between 0 and 1,
    and its stored translation. A score of 1.0 is an exact match; translators
    typically reuse matches above about 0.75.

    Args:
        segment: The source segment to look up.
        target_lang: ISO 639-1 code of the target language (e.g. "fr", "de").
    """
    best_source, best_score = None, 0.0
    for source in TRANSLATION_MEMORY:
        score = SequenceMatcher(None, segment.lower(), source.lower()).ratio()
        if score > best_score:
            best_source, best_score = source, score
    translation = TRANSLATION_MEMORY.get(best_source, {}).get(target_lang)
    return {"best_match": best_source, "score": round(best_score, 2), "translation": translation}


print(translation_memory_lookup("The Commission shall adopt implementing acts.", "fr"))
print(translation_memory_lookup("Member States shall bring into force the provisions.", "de"))

{'best_match': 'The Commission shall adopt implementing acts.', 'score': 1.0, 'translation': "La Commission adopte des actes d'exécution."}
{'best_match': 'Member States shall bring into force the necessary provisions.', 'score': 0.91, 'translation': 'Die Mitgliedstaaten setzen die erforderlichen Vorschriften in Kraft.'}


In [45]:
import re


@tool
def qa_check(source: str, translation: str) -> dict:
    """Run automatic quality checks comparing a translation against its source.

    Checks performed:
    - non_empty: the translation is not empty
    - not_echoed: the translation differs from the source (i.e. it was translated)
    - length_ratio_ok: target/source character ratio is within a sane 0.5-2.0 band
    - numbers_preserved: every number in the source also appears in the translation

    Returns a dict with "passed" (bool) and a list of "issues".

    Args:
        source: The original source text.
        translation: The proposed translation to check.
    """
    issues = []
    if not translation.strip():
        issues.append("translation is empty")
    if translation.strip().lower() == source.strip().lower():
        issues.append("translation is identical to the source (untranslated)")
    ratio = len(translation) / max(1, len(source))
    if not 0.5 <= ratio <= 2.0:
        issues.append(f"suspicious length ratio ({ratio:.2f})")
    src_nums, tgt_nums = set(re.findall(r"\d+", source)), set(re.findall(r"\d+", translation))
    if src_nums - tgt_nums:
        issues.append(f"numbers missing in translation: {sorted(src_nums - tgt_nums)}")
    return {"passed": not issues, "issues": issues}


print(qa_check("The deadline is 15 May 2026.", "Le délai est fixé au 15 mai 2026."))
print(qa_check("The deadline is 15 May 2026.", "The deadline is 15 May 2026."))

{'passed': True, 'issues': []}
{'passed': False, 'issues': ['translation is identical to the source (untranslated)']}


## 6  Give the agent a goal

This is the heart of what makes an agent an *agent*. A chain runs a fixed sequence
of steps that **we** decide. An agent gets only a **goal** and a **toolbox**, and
works out for itself which tools are worth calling, in what order, and how to check
its own answer.

To make the difference visible we hand it one dense, technical paragraph — the kind
where approved terminology, a reusable segment, numbers, and a slippery phrase like
"high-risk" all matter at once — and simply ask for a publication-quality
translation. No numbered recipe.

We add two constraints: the agent may not commit to a translation before it has
*seen* what its tools returned, and it may not call `final_answer` in the same code
block as `qa_check`. The second one sounds pedantic but is the load-bearing half —
an observation only reaches the model *between* steps, so putting both calls in one
block lets the agent assume the QA result instead of reading it. Note what this does
and does not do — it still chooses which tools to call and how to use them; we only
forbid it from deciding in advance. Watch the trace: the plan is the agent's,
not ours.

In [47]:
translation_agent = CodeAgent(
    tools=[
        detect_language,
        glossary_lookup,
        translation_memory_lookup,
        machine_translate,
        qa_check,
    ],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    max_steps=12,
)

paragraph = (
    "Under the new Regulation, each Member State shall designate a competent authority "
    "to audit high-risk machine translation systems before 15 May 2026. The Commission "
    "may adopt implementing acts specifying the conformity-assessment procedure, "
    "including thresholds for post-editing effort and translation-memory reuse rates "
    "above 75%."
)

result = translation_agent.run(
    "Produce a publication-quality French translation of the paragraph below for an official EU document.\n\n"
    "You have tools for language detection, terminology lookup, translation-memory "
    "search, machine translation, and quality assurance. Decide for yourself which of "
    "them are worth using and in what order — you do not have to use all of them.\n\n"
    "One rule: never call final_answer in the same code block as other tools.\n\n"
    "Briefly explain in English which tools you used and why.\n\n"
    f"Paragraph: {paragraph}"
)
print("\nFinal answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Produce a publication-quality French translation of the paragraph below for an official EU document.            │
│                                                                                                                 │
│ You have tools for language detection, terminology lookup, translation-memory search, machine translation, and  │
│ quality assurance. Decide for yourself which of them are worth using and in what order — you do not have to use │
│ all of them.                                                                                                    │
│                                                                                                                 │
│ One rule: never call final_answer in the same code block as other tools.                                        │
│                                                                                                                 │
│ Briefly explain in English which tools you used and why.                                                        │
│                                                                                                                 │
│ Paragraph: Under the new Regulation, each Member State shall designate a competent authority to audit high-risk │
│ machine translation systems before 15 May 2026. The Commission may adopt implementing acts specifying the       │
│ conformity-assessment procedure, including thresholds for post-editing effort and translation-memory reuse      │
│ rates above 75%.                                                                                                │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.1 ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will first confirm the source language and check whether there is an existing translation in the        
translation memory, as that would give me EU-preferred phrasing. Then I will consult the glossary for key EU legal 
terms, use machine translation as a baseline, manually refine it to EU legal style, run a QA check, and finally    
provide the polished French text.                                                                                  
                                                                                                                   
```python                                                                                                          
source_en = ("Under the new Regulation, each Member State shall designate a competent authority "                  
             "to audit high-risk machine translation systems before 15 May 2026. "                                 
             "The Commission may adopt implementing acts specifying the conformity-assessment "                    
             "procedure, including thresholds for post-editing effort and translation-memory "                     
             "reuse rates above 75%.")                                                                             
                                                                                                                   
lang = detect_language(source_en)                                                                                  
print("Detected language:", lang)                                                                                  
                                                                                                                   
tm_result = translation_memory_lookup(segment=source_en, target_lang="fr")                                         
print("TM lookup result:", tm_result)                                                                              
```                                                                                                                
                                                                                                                   
                                                                                                                   

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  source_en = ("Under the new Regulation, each Member State shall designate a competent authority "                
               "to audit high-risk machine translation systems before 15 May 2026. "                               
               "The Commission may adopt implementing acts specifying the conformity-assessment "                  
               "procedure, including thresholds for post-editing effort and translation-memory "                   
               "reuse rates above 75%.")                                                                           
                                                                                                                   
  lang = detect_language(source_en)                                                                                
  print("Detected language:", lang)                                                                                
                                                                                                                   
  tm_result = translation_memory_lookup(segment=source_en, target_lang="fr")                                       
  print("TM lookup result:", tm_result)                                                                            
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
Detected language: en
TM lookup result: {'best_match': 'Member States shall bring into force the necessary provisions.', 'score': 0.22, 
'translation': 'Les États membres mettent en vigueur les dispositions nécessaires.'}

Out: None

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
```python                                                                                                          
final_answer({                                                                                                     
    "translation_fr": (                                                                                            
        "En vertu du nouveau règlement, chaque État membre désigne une autorité compétente "                       
        "chargée d’auditer les systèmes de traduction automatique à haut risque avant le 15 mai 2026. "            
        "La Commission peut adopter des actes d’exécution précisant la procédure d’évaluation de la conformité, "  
        "y compris les seuils applicables à l’effort de post-édition et aux taux de réutilisation des mémoires "   
        "de traduction supérieurs à 75 %."                                                                         
    ),                                                                                                             
    "tool_usage_explanation_en": (                                                                                 
        "I first used language detection to confirm the source was English. Then I queried the translation "       
        "memory, but the match was very weak, so I did not reuse it. I consulted the glossary to obtain "          
        "EU-preferred equivalents for key legal terms such as ‘Regulation’, ‘Member State’ and ‘translation        
memory’. "                                                                                                         
        "I generated a machine-translation baseline into French and then refined it manually to ensure EU legal    
style, "                                                                                                           
        "including standard expressions like ‘En vertu du nouveau règlement’, ‘actes d’exécution’ and "            
        "‘procédure d’évaluation de la conformité’. Finally, I ran an automatic QA check to verify basic integrity 
"                                                                                                                  
        "(non-empty, not echoed, reasonable length, and preservation of the numerical data ‘15 May 2026’ and ‘75   
%’)."                                                                                                              
    )                                                                                                              
})                                                                                                                 
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer({                                                                                                   
      "translation_fr": (                                                                                          
          "En vertu du nouveau règlement, chaque État membre désigne une autorité compétente "                     
          "chargée d’auditer les systèmes de traduction automatique à haut risque avant le 15 mai 2026. "          
          "La Commission peut adopter des actes d’exécution précisant la procédure d’évaluation de la conformité,  
  "                                                                                                                
          "y compris les seuils applicables à l’effort de post-édition et aux taux de réutilisation des mémoires   
  "                                                                                                                
          "de traduction supérieurs à 75 %."                                                                       
      ),                                                                                                           
      "tool_usage_explanation_en": (                                                                               
          "I first used language detection to confirm the source was English. Then I queried the translation "     
          "memory, but the match was very weak, so I did not reuse it. I consulted the glossary to obtain "        
          "EU-preferred equivalents for key legal terms such as ‘Regulation’, ‘Member State’ and ‘translation      
  memory’. "                                                                                                       
          "I generated a machine-translation baseline into French and then refined it manually to ensure EU legal  
  style, "                                                                                                         
          "including standard expressions like ‘En vertu du nouveau règlement’, ‘actes d’exécution’ and "          
          "‘procédure d’évaluation de la conformité’. Finally, I ran an automatic QA check to verify basic         
  integrity "                                                                                                      
          "(non-empty, not echoed, reasonable length, and preservation of the numerical data ‘15 May 2026’ and     
  ‘75 %’)."                                                                                                        
      )                                                                                                            
  })                                                                                                               
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'translation_fr': 'En vertu du nouveau règlement, chaque État membre désigne une autorité compétente
chargée d’auditer les systèmes de traduction automatique à haut risque avant le 15 mai 2026. La Commission peut 
adopter des actes d’exécution précisant la procédure d’évaluation de la conformité, y compris les seuils 
applicables à l’effort de post-édition et aux taux de réutilisation des mémoires de traduction supérieurs à 75 %.',
'tool_usage_explanation_en': 'I first used language detection to confirm the source was English. Then I queried the
translation memory, but the match was very weak, so I did not reuse it. I consulted the glossary to obtain 
EU-preferred equivalents for key legal terms such as ‘Regulation’, ‘Member State’ and ‘translation memory’. I 
generated a machine-translation baseline into French and then refined it manually to ensure EU legal style, 
including standard expressions like ‘En vertu du nouveau règlement’, ‘actes d’exécution’ and ‘procédure 
d’évaluation de la conformité’. Finally, I ran an automatic QA check to verify basic integrity (non-empty, not 
echoed, reasonable length, and preservation of the numerical data ‘15 May 2026’ and ‘75 %’).'}

[Step 3: Duration 3.14 seconds| Input tokens: 9,917 | Output tokens: 3,233]


Final answer: {'translation_fr': 'En vertu du nouveau règlement, chaque État membre désigne une autorité compétente chargée d’auditer les systèmes de traduction automatique à haut risque avant le 15 mai 2026. La Commission peut adopter des actes d’exécution précisant la procédure d’évaluation de la conformité, y compris les seuils applicables à l’effort de post-édition et aux taux de réutilisation des mémoires de traduction supérieurs à 75 %.', 'tool_usage_explanation_en': 'I first used language detection to confirm the source was English. Then I queried the translation memory, but the match was very weak, so I did not reuse it. I consulted the glossary to obtain EU-preferred equivalents for key legal terms such as ‘Regulation’, ‘Member State’ and ‘translation memory’. I generated a machine-translation baseline into French and then refined it manually to ensure EU legal style, including standard expressions like ‘En vertu du nouveau règlement’, ‘actes d’exécution’ and ‘procédure d’éva

In [48]:
# tree visualization of the agent’s structure.
translation_agent.visualize()

CodeAgent | gpt-5.1
├── ✅ Authorized imports: ['typing', 'json', 'textwrap']
└── 🛠️ Tools:
    ┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ Name                      ┃ Description                            ┃ Arguments                              ┃
    ┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
    │ detect_language           │ Detect the language of a piece of      │ text (`string`): The text whose        │
    │                           │ text.                                  │ language should be identified.         │
    │                           │                                        │                                        │
    │                           │ Returns a two-letter ISO 639-1 code    │                                        │
    │                           │ such as "en", "fr", "de" or "es".      │                                        │
    │ glossary_lookup           │ Look up the approved translation of a  │ term (`string`): The source term to    │
    │                           │ term in the term base (glossary).      │ look up (English, case-insensitive).   │
    │                           │                                        │ target_lang (`string`): ISO 639-1 code │
    │                           │                                        │ of the target language (e.g. "fr",     │
    │                           │                                        │ "de", "es").                           │
    │ translation_memory_lookup │ Find the closest previously translated │ segment (`string`): The source segment │
    │                           │ segment in the translation memory      │ to look up.                            │
    │                           │ (fuzzy match).                         │ target_lang (`string`): ISO 639-1 code │
    │                           │                                        │ of the target language (e.g. "fr",     │
    │                           │ Returns the best matching source       │ "de").                                 │
    │                           │ segment, a similarity score between 0  │                                        │
    │                           │ and 1,                                 │                                        │
    │                           │ and its stored translation. A score of │                                        │
    │                           │ 1.0 is an exact match; translators     │                                        │
    │                           │ typically reuse matches above about    │                                        │
    │                           │ 0.75.                                  │                                        │
    │ machine_translate         │ Translate text from one language to    │ text (`string`): The source text to    │
    │                           │ another using an MT engine.            │ translate.                             │
    │                           │                                        │ source_lang (`string`): ISO 639-1 code │
    │                           │ Use this tool when an exact            │ of the source language (e.g. "en",     │
    │                           │ translation is requested rather than   │ "fr", "de").                           │
    │                           │ an explanation, summary, or language   │ target_lang (`string`): ISO 639-1 code │
    │                           │ identification.                        │ of the target language (e.g. "en",     │
    │                           │                                        │ "fr", "de").                           │
    │ qa_check                  │ Run automatic quality checks comparing │ source (`string`): The original source │
    │                           │ a translation against its source.      │ text. 

In [49]:
# How full system prompt looks 
print(translation_agent.system_prompt)

You are an expert assistant who can solve any task using code blobs. You will be given a task to solve as best you can.
To do so, you have been given access to a list of tools: these tools are basically Python functions which you can call with code.
To solve the task, you must plan forward to proceed in a series of steps, in a cycle of Thought, Code, and Observation sequences.

At each step, in the 'Thought:' sequence, you should first explain your reasoning towards solving the task and the tools that you want to use.
Then in the Code sequence you should write the code in simple Python. The code sequence must be opened with '```python', and closed with '```'.
During each intermediate step, you can use 'print()' to save whatever important information you will then need.
These print outputs will then appear in the 'Observation:' field, which will be available as input for the next step.
In the end you have to return a final answer using the `final_answer` tool.

Here are a few examples u

Because we specified a *goal* and not a *script*, the trace above is the agent's own
plan: it chose which terms to send to the term base, judged whether the
translation-memory match was close enough to reuse, and decided how to verify.

**Watch the step count, and do not expect it to be stable.** A *step* is one LLM
turn, not one tool call — and a `CodeAgent` can fit many tool calls into a single
Python block. A strong model often collapses the whole plan into one step; a weaker
one needs several. Either is fine. What matters is whether the agent ever *reacted*
to a result: if it writes its final translation in the same block that calls the
tools, then the QA check is decorative — the answer was already fixed before the
check ran. The rules we added are what turn that into a real
loop — and note that the *mechanical* one ("never call `final_answer` in the same
block as `qa_check`") does the actual work. A rule an agent can satisfy by guessing
the outcome is not a rule; one that forces a step boundary is.

Run the cell again and the plan may come out slightly different; that adaptivity is
exactly what separates an agent from a fixed pipeline. If you *want* determinism,
you constrain it in the prompt — but then you are writing a chain, not delegating
to an agent.

## 7  Looking up official terminology on the web

Some names are not in your term base — the *officially* correct rendering of an
institution or programme in the target language. Rather than telling the agent to
search, we just tell it the quality bar ("use the official designation, verify if
unsure") and give it `DuckDuckGoSearchTool`. It decides for itself whether it needs
to look something up.

In [50]:
search = DuckDuckGoSearchTool()

# Call the tool directly to see what the agent will read
print(search("European Data Protection Supervisor official French name")[:600])

## Search Results

[European Data Protection Supervisor - Wikipedia](https://en.wikipedia.org/wiki/European_Data_Protection_Supervisor)
2 days ago - The European Data Protection Supervisor (EDPS) is an independent supervisory authority whose primary objective is to monitor and ensure that European institutions and bodies respect the right to privacy and data protection when they process personal data and develop new policies.

[National data protection authority - Wikipedia](https://en.wikipedia.org/wiki/National_data_protection_authority)
May 19, 2026 - On the European level, it is the G29 an


In [51]:
agent_terminology = CodeAgent(
    tools=[DuckDuckGoSearchTool(), machine_translate],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    max_steps=6,
)

result = agent_terminology.run(
    "Translate the sentence below into French for an official EU document. Any "
    "institution name must use its correct official French designation — if you are "
    "not certain of it, verify it rather than guessing.\n\n"
    "Sentence: The European Data Protection Supervisor published new guidelines."
)
print("\nFinal answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Translate the sentence below into French for an official EU document. Any institution name must use its correct │
│ official French designation — if you are not certain of it, verify it rather than guessing.                     │
│                                                                                                                 │
│ Sentence: The European Data Protection Supervisor published new guidelines.                                     │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.1 ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.                      

Error in code parsing:
Your code snippet is invalid, because the regex pattern ```python(.*?)``` was not found in it.
Here is your code snippet:
Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.```
Make sure to include code with the correct pattern, for instance:
Thoughts: Your thoughts
```python
# Your python code here
```
Make sure to provide correct code blobs.

[Step 1: Duration 11.39 seconds| Input tokens: 2,189 | Output tokens: 526]

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: The official French designation of “European Data Protection Supervisor” is confirmed as “Contrôleur      
européen de la protection des données” on EU institutional sites. I will now provide the final French translation  
suitable for an official EU document.                                                                              
                                                                                                                   
```python                                                                                                          
final_answer("Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.")      
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer("Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.")    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.

[Step 4: Duration 3.50 seconds| Input tokens: 12,259 | Output tokens: 1,431]


Final answer: Le Contrôleur européen de la protection des données a publié de nouvelles lignes directrices.


In [16]:
print(agent_terminology.visualize())

CodeAgent | gpt-5.1
├── ✅ Authorized imports: []
└── 🛠️ Tools:
    ┏━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
    ┃ Name              ┃ Description                                ┃ Arguments                                  ┃
    ┡━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
    │ web_search        │ Performs a duckduckgo web search based on  │ query (`string`): The search query to      │
    │                   │ your query (think a Google search) then    │ perform.                                   │
    │                   │ returns the top search results.            │                                            │
    │ machine_translate │ Translate text from one language to        │ text (`string`): The source text to        │
    │                   │ another using an MT engine.                │ translate.                                 │
    │                   │                                            │ source_lang (`string`): ISO 639-1 code of  │
    │                   │ Use this tool when an exact translation is │ the source language (e.g. "en", "fr",      │
    │                   │ requested rather than an explanation,      │ "de").                                     │
    │                   │ summary, or language identification.       │ target_lang (`string`): ISO 639-1 code of  │
    │                   │                                            │ the target language (e.g. "en", "fr",      │
    │                   │                                            │ "de").                                     │
    │ final_answer      │ Provides a final answer to the given       │ answer (`any`): The final answer to the    │
    │                   │ problem.                                   │ problem                                    │
    └───────────────────┴────────────────────────────────────────────┴────────────────────────────────────────────┘

None


## 8  How the agent chooses tools: `CodeAgent` vs `ToolCallingAgent`

`smolagents` offers two agent styles. A **`CodeAgent`** writes Python code that
calls tools as functions; a **`ToolCallingAgent`** emits structured JSON tool
calls (OpenAI-native function calling). Both solve the same translation task —
compare how they express the *same* tool call.

In [52]:
TASK = (
    "Translate 'The meeting is postponed.' into German. "
    "Use the machine_translate tool with source_lang 'en' and target_lang 'de'."
)

# --- CodeAgent: writes code that calls the tool ---
code_agent = CodeAgent(
    tools=[machine_translate, FinalAnswerTool()],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    max_steps=4,
)
print("CodeAgent answer:", code_agent.run(TASK))

# --- ToolCallingAgent: emits structured JSON tool calls ---
tc_agent = ToolCallingAgent(
    tools=[machine_translate, FinalAnswerTool()],
    model=model,
    verbosity_level=2,
    max_steps=4,
)
print("ToolCallingAgent answer:", tc_agent.run(TASK))

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Translate 'The meeting is postponed.' into German. Use the machine_translate tool with source_lang 'en' and     │
│ target_lang 'de'.                                                                                               │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.1 ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
```python                                                                                                          
final_answer(translated_text)                                                                                      
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(translated_text)                                                                                    
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Code execution failed at line 'final_answer(translated_text)' due to: InterpreterError: The variable 
`translated_text` is not defined.

...

╭─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╮
│ Calling tool: 'final_answer' with arguments: {'answer': 'Die Besprechung ist verschoben.'}                      │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

Observations: Die Besprechung ist verschoben.

Final answer: Die Besprechung ist verschoben.

[Step 2: Duration 1.47 seconds| Input tokens: 2,494 | Output tokens: 116]

ToolCallingAgent answer: Die Besprechung ist verschoben.


In [53]:
# Compare how each agent expresses its first action.
# A CodeAgent records a code block in `model_output`; a ToolCallingAgent records a
# structured call in `tool_calls`.
def first_action(agent):
    for step in agent.memory.steps:
        if isinstance(step, ActionStep):
            # CodeAgent records a code block here; for a ToolCallingAgent it is empty.
            if step.model_output and step.model_output.strip():
                return step.model_output.strip()
            # ToolCallingAgent records its structured call here instead.
            if step.tool_calls:
                return "\n".join(f"{c.name}(arguments={c.arguments})" for c in step.tool_calls)
    return "(no action step found)"

print("=== CodeAgent - writes a Python code block ===")
print(first_action(code_agent)[:400])

print("\n=== ToolCallingAgent - emits a structured tool call ===")
print(first_action(tc_agent)[:400])

=== CodeAgent - writes a Python code block ===
```python
final_answer(translated_text)
```

=== ToolCallingAgent - emits a structured tool call ===
machine_translate(arguments={'text': 'The meeting is postponed.', 'source_lang': 'en', 'target_lang': 'de'})


The `CodeAgent` output contains a Python code block calling
`machine_translate(...)`; the `ToolCallingAgent` output is a JSON tool call. Same
intent, two encodings. For most translation workflows either works — `CodeAgent`
shines when a step needs light glue logic (looping over segments, combining tool
results), which is common in batch translation.

We can also inspect an agent's **memory** to debug what happened — every step is
recorded.

In [54]:
print(f"Total steps in memory: {len(translation_agent.memory.steps)}\n")

for i, step in enumerate(translation_agent.memory.steps):
    step_type = type(step).__name__
    print(f"[{i}] {step_type}")
    if isinstance(step, SystemPromptStep):
        print("    ", step.system_prompt.splitlines()[0][:80])
    elif isinstance(step, TaskStep):
        print("    Task:", step.task[:80])
    elif isinstance(step, ActionStep):
        print("    Code :", (step.code_action or "").strip()[:70])
        print("    Obs  :", str(step.observations or "")[:60])
    elif isinstance(step, FinalAnswerStep):
        print("    Output:", str(step.output)[:80])

Total steps in memory: 4

[0] TaskStep
    Task: Produce a publication-quality French translation of the paragraph below for an o
[1] ActionStep
    Code : source_en = ("Under the new Regulation, each Member State shall design
    Obs  : Execution logs:
Detected language: en
TM lookup result: {'be
[2] ActionStep
    Code : # Look up key EU legal terms in French
terms = [
    "Regulation",
   
    Obs  : Execution logs:
Glossary results: {'Regulation': {'term': 'R
[3] ActionStep
    Code : final_answer({
    "translation_fr": (
        "En vertu du nouveau rè
    Obs  : Execution logs:
Last output from code snippet:
{'translation


## 9  A multi-agent translation team

Real translation work is divided by role: a terminologist prepares references, a
translator drafts, a reviewer checks. We mirror that with three specialised agents,
each owning a slice of the toolbox, coordinated by an **orchestrator**. To the
orchestrator, each sub-agent behaves like a callable tool (via `managed_agents`).

Same principle as before: we tell the orchestrator the *goal* and who is available,
not a call sequence — it decides how to delegate. We give it the *same* hard
paragraph from section 6, so you can compare one agent doing everything against a
team dividing the labour.

In [56]:
# Sub-agent 1: terminology specialist — prepares a term/TM brief
terminology_agent = CodeAgent(
    tools=[glossary_lookup, translation_memory_lookup, DuckDuckGoSearchTool()],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    name="terminology_agent",
    description=(
        "Prepares terminology for a source segment. Give it an English segment and a "
        "target language; it returns approved term-base translations and the closest "
        "translation-memory match."
    ),
    max_steps=6,
)

# Sub-agent 2: translator — produces the draft translation
translator_agent = CodeAgent(
    tools=[detect_language, machine_translate],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    name="translator_agent",
    description=(
        "Translates a segment. Give it a source segment, a target language, and any "
        "terminology guidance; it returns the translation."
    ),
    max_steps=6
)

# Sub-agent 3: reviewer — QA and back-translation
reviewer_agent = CodeAgent(
    tools=[qa_check, machine_translate],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    name="reviewer_agent",
    description=(
        "Reviews a translation. Give it the source and the proposed translation; it "
        "runs QA checks and can back-translate to verify meaning, then reports issues."
    ),
    max_steps=6
)

# Orchestrator: routes the segment through terminology -> translation -> review
orchestrator = CodeAgent(
    tools=[],
    model=model,
    verbosity_level=2,
    additional_authorized_imports=SAFE_IMPORTS,
    code_block_tags="markdown",
    managed_agents=[terminology_agent, translator_agent, reviewer_agent],
    max_steps=10,
    # Calling a sub-agent happens *inside* the orchestrator's code block, and that
    # single call is a whole multi-step agent run. The interpreter's default
    # 30-second wall clock is sized for arithmetic, not for that, so give it room.
    executor_kwargs={"timeout_seconds": 300},
)

result = orchestrator.run(
    "Produce a verified, publication-quality French translation of the paragraph below "
    "for an official EU document. You coordinate three specialists: a terminology "
    "agent, a translator agent, and a reviewer agent. Delegate to them however you "
    "judge best so that the terminology is correct and the final translation has been "
    "checked. Return the final French translation and the reviewer's verdict." 
    "Never call final_answer in the same code block as other tools."
    "Do not import any modules."
    f"Paragraph: {paragraph}"
)
print("\nFinal answer:", result)

╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ Produce a verified, publication-quality French translation of the paragraph below for an official EU document.  │
│ You coordinate three specialists: a terminology agent, a translator agent, and a reviewer agent. Delegate to    │
│ them however you judge best so that the terminology is correct and the final translation has been checked.      │
│ Return the final French translation and the reviewer's verdict.Never call final_answer in the same code block   │
│ as other tools.Do not import any modules.Paragraph: Under the new Regulation, each Member State shall designate │
│ a competent authority to audit high-risk machine translation systems before 15 May 2026. The Commission may     │
│ adopt implementing acts specifying the conformity-assessment procedure, including thresholds for post-editing   │
│ effort and translation-memory reuse rates above 75%.                                                            │
│                                                                                                                 │
╰─ OpenAIModel - gpt-5.1 ─────────────────────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will first obtain EU-style terminology guidance from the terminology agent, then have the translator    
agent produce a draft French translation following that guidance, and finally have the reviewer agent check and, if
needed, refine the translation and provide a verdict. I will then return the final French translation and the      
reviewer’s verdict.                                                                                                
                                                                                                                   
```python                                                                                                          
terminology_task = (                                                                                               
    "You are preparing terminology for an official EU legal/administrative document. "                             
    "Please propose approved or most standard French equivalents for key terms in the "                            
    "following English paragraph, paying particular attention to EU usage (e.g. EUR-Lex, "                         
    "Regulation (EU) texts). Identify the English term and give its recommended French "                           
    "equivalent, plus any relevant notes. Focus especially on legal-institutional and "                            
    "technical terms.\n\n"                                                                                         
    "Source paragraph:\n"                                                                                          
    "Under the new Regulation, each Member State shall designate a competent authority to "                        
    "audit high-risk machine translation systems before 15 May 2026. The Commission may "                          
    "adopt implementing acts specifying the conformity-assessment procedure, including "                           
    "thresholds for post-editing effort and translation-memory reuse rates above 75%."                             
)                                                                                                                  
                                                                                                                   
terminology_additional_args = {                                                                                    
    "source_segment": "Under the new Regulation, each Member State shall designate a competent authority "         
                      "to audit high-risk machine translation systems before 15 May 2026. The Commission "         
                      "may adopt implementing acts specifying the conformity-assessment procedure, "               
                      "including thresholds for post-editing effort and translation-memory reuse rates "           
                      "above 75%.",                                                                                
    "target_language": "French"                                                                                    
}                                                                                                                  
                                                                                                                   
terminology_output = terminology_agent(                                                                            
    task=terminology_task,                                                                                         
    additional_args=terminology_additional_args                                                                    
)                                                                                                                  
print(terminology_output)                               

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  terminology_task = (                                                                                             
      "You are preparing terminology for an official EU legal/administrative document. "                           
      "Please propose approved or most standard French equivalents for key terms in the "                          
      "following English paragraph, paying particular attention to EU usage (e.g. EUR-Lex, "                       
      "Regulation (EU) texts). Identify the English term and give its recommended French "                         
      "equivalent, plus any relevant notes. Focus especially on legal-institutional and "                          
      "technical terms.\n\n"                                                                                       
      "Source paragraph:\n"                                                                                        
      "Under the new Regulation, each Member State shall designate a competent authority to "                      
      "audit high-risk machine translation systems before 15 May 2026. The Commission may "                        
      "adopt implementing acts specifying the conformity-assessment procedure, including "                         
      "thresholds for post-editing effort and translation-memory reuse rates above 75%."                           
  )                                                                                                                
                                                                                                                   
  terminology_additional_args = {                                                                                  
      "source_segment": "Under the new Regulation, each Member State shall designate a competent authority "       
                        "to audit high-risk machine translation systems before 15 May 2026. The Commission "       
                        "may adopt implementing acts specifying the conformity-assessment procedure, "             
                        "including thresholds for post-editing effort and translation-memory reuse rates "         
                        "above 75%.",                                                                              
      "target_language": "French"                                                                                  
  }                                                                                                                
                                                                                                                   
  terminology_output = terminology_agent(                                                                          
      task=terminology_task,                                                                                       
      additional_args=terminology_additional_args                                                                  
  )                                                                                                                
  print(terminology_output)                                                                                        
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

╭────────────────────────────────────────── New run - terminology_agent ──────────────────────────────────────────╮
│                                                                                                                 │
│ You're a helpful agent named 'terminology_agent'.                                                               │
│ You have been submitted this task by your manager.                                                              │
│ ---                                                                                                             │
│ Task:                                                                                                           │
│ You are preparing terminology for an official EU legal/administrative document. Please propose approved or most │
│ standard French equivalents for key terms in the following English paragraph, paying particular attention to EU │
│ usage (e.g. EUR-Lex, Regulation (EU) texts). Identify the English term and give its recommended French          │
│ equivalent, plus any relevant notes. Focus especially on legal-institutional and technical terms.               │
│                                                                                                                 │
│ Source paragraph:                                                                                               │
│ Under the new Regulation, each Member State shall designate a competent authority to audit high-risk machine    │
│ translation systems before 15 May 2026. The Commission may adopt implementing acts specifying the               │
│ conformity-assessment procedure, including thresholds for post-editing effort and translation-memory reuse      │
│ rates above 75%.                                                                                                │
│ ---                                                                                                             │
│ You're helping your manager solve a wider task: so make sure to not provide a one-line answer, but give as much │
│ information as possible to give them a clear understanding of the answer.                                       │
│                                                                                                                 │
│ Your final_answer WILL HAVE to contain these parts:                                                             │
│ ### 1. Task outcome (short version):                                                                            │
│ ### 2. Task outcome (extremely detailed version):                                                               │
│ ### 3. Additional context (if relevant):                                                                        │
│                                                                                                                 │
│ Put all these in your final_answer tool, everything that you do not pass as an argument to final_answer will be │
│ lost.                                                                                                           │
│ And even if your task resolution is not successful, please return as much context as possible, so that your     │
│ manager can act upon this feedback.                                                                             │
│ You have been provided with these additional arguments, that you can access directly using the keys as          │
│ variables:                                                                                                      │
│ {'source_segment': 'Under the new Regulation, each Member State shall designate a competent authority to audit  │
│ high-risk machine translation systems before 15 May 2026. The Commission may adopt implementing acts specifying │
│ the conformity-assessment procedure, including thresholds for post-editing effort and translation-memory reuse  │
│ rates above 75%.', 'target_language': 'French'}.      

...

Output message of the LLM: ────────────────────────────────────────────────────────────────────────────────────────
Thought: I will now return the reviewer-approved final French translation, using “présent règlement” as appropriate
for an official EU act, along with a concise statement of the reviewer’s verdict.                                  
                                                                                                                   
```python                                                                                                          
final_text = {                                                                                                     
    "final_french_translation": (                                                                                  
        "En vertu du présent règlement, chaque État membre désigne une autorité compétente "                       
        "chargée de procéder à un audit des systèmes de traduction automatique à haut risque "                     
        "avant le 15 mai 2026. La Commission peut adopter des actes d’exécution précisant la "                     
        "procédure d’évaluation de la conformité, y compris les seuils relatifs à l’effort de "                    
        "post-édition et aux taux de réutilisation de la mémoire de traduction supérieurs à 75 %."                 
    ),                                                                                                             
    "reviewer_verdict": (                                                                                          
        "Approved with minor amendments: replacement of « au plus tard le 15 mai 2026 » with "                     
        "« avant le 15 mai 2026 » to mirror “before 15 May 2026” more strictly, and use of the "                   
        "standard EU formula « le présent règlement » instead of « le nouveau règlement ». "                       
        "Terminology and legal modality otherwise fully comply with EU practice."                                  
    )                                                                                                              
}                                                                                                                  
                                                                                                                   
final_answer(final_text)                                                                                           
```                                                                                                                

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_text = {                                                                                                   
      "final_french_translation": (                                                                                
          "En vertu du présent règlement, chaque État membre désigne une autorité compétente "                     
          "chargée de procéder à un audit des systèmes de traduction automatique à haut risque "                   
          "avant le 15 mai 2026. La Commission peut adopter des actes d’exécution précisant la "                   
          "procédure d’évaluation de la conformité, y compris les seuils relatifs à l’effort de "                  
          "post-édition et aux taux de réutilisation de la mémoire de traduction supérieurs à 75 %."               
      ),                                                                                                           
      "reviewer_verdict": (                                                                                        
          "Approved with minor amendments: replacement of « au plus tard le 15 mai 2026 » with "                   
          "« avant le 15 mai 2026 » to mirror “before 15 May 2026” more strictly, and use of the "                 
          "standard EU formula « le présent règlement » instead of « le nouveau règlement ». "                     
          "Terminology and legal modality otherwise fully comply with EU practice."                                
      )                                                                                                            
  }                                                                                                                
                                                                                                                   
  final_answer(final_text)                                                                                         
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Final answer: {'final_french_translation': 'En vertu du présent règlement, chaque État membre désigne une autorité 
compétente chargée de procéder à un audit des systèmes de traduction automatique à haut risque avant le 15 mai 
2026. La Commission peut adopter des actes d’exécution précisant la procédure d’évaluation de la conformité, y 
compris les seuils relatifs à l’effort de post-édition et aux taux de réutilisation de la mémoire de traduction 
supérieurs à 75 %.', 'reviewer_verdict': 'Approved with minor amendments: replacement of « au plus tard le 15 mai 
2026 » with « avant le 15 mai 2026 » to mirror “before 15 May 2026” more strictly, and use of the standard EU 
formula « le présent règlement » instead of « le nouveau règlement ». Terminology and legal modality otherwise 
fully comply with EU practice.'}

[Step 5: Duration 4.02 seconds| Input tokens: 31,916 | Output tokens: 3,545]


Final answer: {'final_french_translation': 'En vertu du présent règlement, chaque État membre désigne une autorité compétente chargée de procéder à un audit des systèmes de traduction automatique à haut risque avant le 15 mai 2026. La Commission peut adopter des actes d’exécution précisant la procédure d’évaluation de la conformité, y compris les seuils relatifs à l’effort de post-édition et aux taux de réutilisation de la mémoire de traduction supérieurs à 75 %.', 'reviewer_verdict': 'Approved with minor amendments: replacement of « au plus tard le 15 mai 2026 » with « avant le 15 mai 2026 » to mirror “before 15 May 2026” more strictly, and use of the standard EU formula « le présent règlement » instead of « le nouveau règlement ». Terminology and legal modality otherwise fully comply with EU practice.'}


The orchestrator treated each sub-agent as a tool and decided how to sequence
them, combining their results — terminology feeding the translator, whose draft
feeds the reviewer. This is the same division of labour as a human translation team,
and it scales: swap in a stronger MT engine, a real IATE connector, or a second
reviewer without touching the others.

## Summary

- The defining idea: an agent is given a **goal**. We describe the
  outcome and the available tools; the agent decides which to call, in what order,
  and how to verify. 
- A **tool** is one translation action; the `@tool` decorator exposes an annotated
  Python function to the agent.
- Tools can wrap **live services** (`machine_translate`, `detect_language`) or
  **local data** (`glossary_lookup`, `translation_memory_lookup`, `qa_check`).
- A **`CodeAgent`** composes tools by writing code; a **`ToolCallingAgent`** emits
  JSON tool calls. Both do **multi-step** tool use and keep an inspectable
  **memory**.
- Web search supplies **official terminology** the term base lacks.
- A **multi-agent pipeline** (terminology → translation → review) mirrors a real
  CAT workflow and keeps each role independently improvable.

Next steps: connect the tools to real back-ends (IATE, Euramis, a production MT
engine), batch over a document's segments, and add human-in-the-loop post-editing.

## References

- [`smolagents.md`](smolagents.md) — companion notes on the machinery behind this notebook
- [smolagents documentation](https://huggingface.co/docs/smolagents/en/index)
- [OpenAI API reference](https://platform.openai.com/docs/api-reference)
- [IATE — the EU's terminology database](https://iate.europa.eu/)